# PhysAR-IDS -- NB2 (v6): XGBoost Optuna TPE -- multiclass, multiseed

**Purpose.** Tune XGBoost on the v6 feature set (NB1 v6) with Optuna's TPE sampler, then report the tuned model against the
untuned baseline **under identical conditions** (same backend, same 5 seeds, paired statistics). The output is the tuned
hyper-parameter set and tuned model that the attack / defence notebooks load.

**Inputs:** `phase1_data_v6.pkl`, `baseline_results_v6.pkl` (from NB1 v6) in `SmartGrid_Dissertation`.
(Set `PIPELINE_VERSION = 'v6min21'` in the setup cell to run the same search on the 21-feature variant.)

## Design decisions (and why)

| Decision | Reason |
|---|---|
| **Selection uses time-blocked CV inside the TRAIN data only** (5 chronological folds per (class, PLC) series, embargo at each cut) | The test split is never used to pick hyper-parameters. An earlier single-split search chose parameters that scored *worse* on test than the baseline -- a single-holdout overfit. |
| **TPE sampler (seed 42) + median pruner on the fold sequence** | TPE as requested; the pruner stops a trial early when its running fold-mean is below the median of earlier trials, saving compute without changing which configurations are considered good. |
| **Search space includes strong regularisers, with `min_child_weight` up to 1000** | Each distinct measurement is repeated ~229x in the row-level data, so a leaf needs a large hessian mass to cover more than one burst; the old 1-20 range could not regularise at the right scale. |
| **Baseline is re-fitted here on the same backend and seeds as the tuned model** | CPU and GPU XGBoost are not bit-identical even with the same `random_state`; comparing a GPU-tuned model with the CPU baseline from NB1 would confound tuning with backend. The NB1 CPU baseline is still printed for reference. |
| **Both models are evaluated on 5 seeds with a paired t-test** | n=5 cannot support a Wilcoxon test below p=0.0625, and a single seed (the old 0.7110 vs 0.7225) is not evidence of anything. |
| **No rule that "tuned must win"** | The result is reported whichever way it falls. If tuning does not beat the baseline, the dissertation says so. |

**What to expect.** NB1 v6 XGBoost scores about 0.85 (CPU). Earlier tuning (v4) only tied the baseline, and the cross-validation
hyper-parameter sweep for v6 was flat. A gain of 0 to +0.02 is plausible; a large jump would be suspicious.

## 0. Setup

In [ ]:
import os, sys, random, time, json, platform, subprocess, warnings
import numpy as np
import pandas as pd
import joblib
from scipy import stats
warnings.filterwarnings('ignore')
try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'])
    import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import xgboost as xgb
optuna.logging.set_verbosity(optuna.logging.WARNING)

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'            # 'v6min21' for the 21-feature variant
SEARCH_SEED      = 42              # TPE sampler seed
SEEDS            = [42, 7, 123, 2024, 99]
N_TRIALS         = 60
N_CV_FOLDS       = 5
EMBARGO_SECONDS  = 1.0

random.seed(SEARCH_SEED); np.random.seed(SEARCH_SEED); os.environ['PYTHONHASHSEED'] = str(SEARCH_SEED)

# GPU detection (XGBoost silently falls back to CPU, so check the warning rather than relying on an exception)
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2)
    _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
if _cpu:
    BACKEND = {'tree_method': 'hist'}
print(f'xgboost {xgb.__version__} | optuna {optuna.__version__} | backend: {BACKEND_NAME}')
print(f'Working directory: {os.getcwd()}')

## 1. Load NB1 v6 data (with version check)

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION, f"version mismatch: {data.get('PIPELINE_VERSION')}"
assert data.get('TEST_FRAC') == 0.15, data.get('TEST_FRAC')
print(f"Version check passed: {PIPELINE_VERSION}, TEST_FRAC={data['TEST_FRAC']}, EMBARGO_SECONDS={data['EMBARGO_SECONDS']}")

X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = data['FEATURE_COLS']; PLC_train = np.asarray(data['PLCID_train']); ts_train = np.asarray(data['timestamp_train'])
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
print(f'Train {X_train.shape}  Test {X_test.shape}  features={len(FEATURE_COLS)}')
print('Train class counts:', np.bincount(y_train), '| Test class counts:', np.bincount(y_test))

## 2. Time-blocked, session-aware CV folds (train data only)

Within every `(class, PLC)` series the timeline is cut into 5 equal chronological blocks; rows within `EMBARGO_SECONDS`
of an internal cut are dropped from all folds. Each fold is validated on a *later* time block than part of its training
data would be in a plain K-fold, but unlike an expanding window every fold has the same size, which keeps the
objective's variance low. (Physical samples arrive ~5 s apart, so the 1 s embargo is non-binding, as shown in NB1.)

In [ ]:
def assign_temporal_cv_folds(label_arr, plc_arr, ts_arr, n_folds=5, embargo=1.0):
    df = pd.DataFrame({'label': label_arr, 'plc': plc_arr, 't': ts_arr})
    fold = np.full(len(df), -1, dtype=int)
    for _, idx in df.groupby(['label', 'plc']).indices.items():
        t = df['t'].values[idx]
        t0, t1 = t.min(), t.max()
        edges = np.linspace(t0, t1, n_folds + 1)
        f = np.clip(np.searchsorted(edges, t, side='right') - 1, 0, n_folds - 1)
        d = np.min(np.abs(t[:, None] - edges[1:-1][None, :]), axis=1) if n_folds > 1 else np.full(len(t), np.inf)
        f[d < embargo] = -1
        fold[idx] = f
    return fold

cv_folds = assign_temporal_cv_folds(y_train, PLC_train, ts_train, N_CV_FOLDS, EMBARGO_SECONDS)
print('Rows per fold:', {int(k): int(v) for k, v in zip(*np.unique(cv_folds, return_counts=True))}, '(-1 = embargoed)')
for f in range(N_CV_FOLDS):
    print(f'  fold {f} class counts:', np.bincount(y_train[cv_folds == f], minlength=6))

## 3. Reference: baseline configuration scored with the same CV (so tuning is judged against it)

In [ ]:
BASELINE_PARAMS = dict(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8)
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

def cv_score(params, seed=SEARCH_SEED, trial=None):
    scores = []
    for f in range(N_CV_FOLDS):
        tr = (cv_folds != f) & (cv_folds != -1); va = (cv_folds == f)
        m = xgb.XGBClassifier(**params, **COMMON, random_state=seed)
        m.fit(X_train[tr], y_train[tr], verbose=False)
        scores.append(f1_score(y_train[va], m.predict(X_train[va]), average='weighted'))
        if trial is not None:
            trial.report(float(np.mean(scores)), step=f)
            if trial.should_prune():
                import optuna as _o; raise _o.TrialPruned()
    return float(np.mean(scores)), scores

t0 = time.time()
base_cv_mean, base_cv_folds = cv_score(BASELINE_PARAMS)
print(f'Baseline config, {N_CV_FOLDS}-fold temporal CV: mean weighted F1 = {base_cv_mean:.4f}  '
      f'(folds: {[round(s, 4) for s in base_cv_folds]})   [{time.time() - t0:.0f}s for {N_CV_FOLDS} fits]')
print(f'Estimated search time: ~{(time.time() - t0) * N_TRIALS / 60 * 1.3:.0f} min for {N_TRIALS} unpruned trials (pruning will reduce this).')

## 4. Optuna TPE search

Objective = mean weighted F1 over the 5 temporal folds. Search space (all ranges inclusive):

| Parameter | Range | Scale |
|---|---|---|
| `n_estimators` | 100 - 600 | int |
| `max_depth` | 3 - 10 | int |
| `learning_rate` | 0.01 - 0.3 | log |
| `subsample` | 0.5 - 1.0 | float |
| `colsample_bytree` | 0.3 - 1.0 | float |
| `reg_alpha`, `reg_lambda` | 1e-4 - 10 | log |
| `min_child_weight` | 1 - 1000 | int, log |
| `gamma` | 1e-4 - 5 | log |

The default configuration is enqueued as trial 0, so the search can never end up worse than the baseline *on the objective*.

In [ ]:
def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 1000, log=True),
        'gamma':            trial.suggest_float('gamma', 1e-4, 5.0, log=True),
    }
    mean, folds = cv_score(params, trial=trial)
    trial.set_user_attr('fold_scores', [float(s) for s in folds])
    return mean

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEARCH_SEED),
                            pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=1),
                            study_name=f'xgb_{PIPELINE_VERSION}')
# trial 0 = the baseline configuration (with regulariser values that are effectively "off")
study.enqueue_trial({**BASELINE_PARAMS, 'reg_alpha': 1e-4, 'reg_lambda': 1.0, 'min_child_weight': 1, 'gamma': 1e-4})

t0 = time.time()
def _cb(study, trial):
    if trial.number % 5 == 0 or trial.number == N_TRIALS - 1:
        done = [t for t in study.trials if t.value is not None]
        print(f'  trial {trial.number:>3}  state={trial.state.name:<9} value={trial.value if trial.value is not None else float("nan"):.4f}  '
              f'best={study.best_value:.4f}  elapsed={(time.time() - t0) / 60:.1f} min  ({len(done)} complete)')
study.optimize(xgb_objective, n_trials=N_TRIALS, callbacks=[_cb])

print(f'\nSearch finished in {(time.time() - t0) / 60:.1f} min.  Best mean CV F1: {study.best_value:.4f}  (baseline config: {base_cv_mean:.4f})')
print('Best params:'); [print(f'  {k}: {v}') for k, v in study.best_params.items()]

## 5. Search diagnostics (is the landscape informative or flat?)

In [ ]:
trials = study.trials_dataframe(attrs=('number', 'value', 'state', 'params', 'user_attrs'))
trials.to_csv(f'optuna_xgb_trials_{PIPELINE_VERSION}.csv', index=False)
done = trials[trials.state == 'COMPLETE'].sort_values('value', ascending=False)
print(f'Trials: {len(trials)} total, {len(done)} complete, {(trials.state == "PRUNED").sum()} pruned')
print('\nTop 8 trials by CV F1:')
print(done.head(8)[['number', 'value'] + [c for c in done.columns if c.startswith('params_')]].round(4).to_string(index=False))

best = study.best_value
print(f'\nTrials within 0.005 of the best CV score: {int((done.value >= best - 0.005).sum())} of {len(done)}')
print(f'Spread of the top-10 CV scores: {done.value.head(10).max() - done.value.head(10).min():.4f}')
print(f'Improvement of best over baseline config on the CV objective: {best - base_cv_mean:+.4f}')
fold_sd = np.std(study.best_trial.user_attrs["fold_scores"], ddof=1)
print(f'Std across folds for the best trial: {fold_sd:.4f}  -> a CV gain smaller than this is not distinguishable from fold noise')

try:
    imp = optuna.importance.get_param_importances(study)
    print('\nHyper-parameter importance (fANOVA):'); [print(f'  {k:<18} {v:.3f}') for k, v in imp.items()]
except Exception as e:
    imp = {}; print('importance not available:', e)

import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
vals = [t.value for t in study.trials if t.value is not None]; nums = [t.number for t in study.trials if t.value is not None]
ax[0].scatter(nums, vals, s=14, alpha=0.7); ax[0].plot(nums, np.maximum.accumulate(vals), color='C3', lw=1.5, label='best so far')
ax[0].axhline(base_cv_mean, ls='--', color='gray', label='baseline config'); ax[0].set_xlabel('trial'); ax[0].set_ylabel('mean CV weighted F1'); ax[0].legend()
ax[0].set_title('Optimisation history')
if imp: ax[1].barh(list(imp.keys())[::-1], list(imp.values())[::-1]); ax[1].set_title('Hyper-parameter importance')
plt.tight_layout(); plt.savefig(f'optuna_xgb_diagnostics_{PIPELINE_VERSION}.png', dpi=150); plt.close()
print(f'Saved optuna_xgb_diagnostics_{PIPELINE_VERSION}.png')

## 6. Held-out test: tuned vs baseline, same backend, 5 seeds each

The tuned parameters were fixed in Section 4 using training data only; the test set is evaluated once here. Both configurations
are refit on the full training set for each seed.

In [ ]:
best_params = dict(study.best_params)

def run_metrics(y_true, y_pred, y_tr_true=None, y_tr_pred=None):
    out = {'test_f1': f1_score(y_true, y_pred, average='weighted'), 'test_macro_f1': f1_score(y_true, y_pred, average='macro'),
           'per_class_f1': f1_score(y_true, y_pred, average=None, labels=range(6), zero_division=0)}
    b = (y_true == 0); a = ~b
    out['benign_fpr'] = float((y_pred[b] != 0).mean()); out['attack_miss_rate'] = float((y_pred[a] == 0).mean())
    if y_tr_true is not None:
        out['train_f1'] = f1_score(y_tr_true, y_tr_pred, average='weighted'); out['gap'] = out['train_f1'] - out['test_f1']
    return out

RESULTS = {'baseline_same_backend': {}, 'tuned': {}}; MODELS = {}
for name, p in [('baseline_same_backend', BASELINE_PARAMS), ('tuned', best_params)]:
    for seed in SEEDS:
        t0 = time.time()
        m = xgb.XGBClassifier(**p, **COMMON, random_state=seed); m.fit(X_train, y_train, verbose=False)
        r = run_metrics(y_test, m.predict(X_test), y_train, m.predict(X_train)); RESULTS[name][seed] = r; MODELS[(name, seed)] = m
        print(f'{name:<22} seed={seed:<5} train={r["train_f1"]:.4f} test={r["test_f1"]:.4f} gap={r["gap"]:.4f} ({time.time() - t0:.0f}s)')

## 7. Summary with variation across seeds + paired test

In [ ]:
T4 = stats.t.ppf(0.975, df=len(SEEDS) - 1)
arr = lambda name, key: np.array([RESULTS[name][s][key] for s in SEEDS])
rows = []
for name in RESULTS:
    te = arr(name, 'test_f1')
    rows.append({'Model': name, 'Test F1 (weighted)': f'{te.mean():.4f} +/- {te.std(ddof=1):.4f}',
                 '95% CI': f'[{te.mean() - T4 * te.std(ddof=1) / np.sqrt(len(te)):.4f}, {te.mean() + T4 * te.std(ddof=1) / np.sqrt(len(te)):.4f}]',
                 'Macro F1': f'{arr(name, "test_macro_f1").mean():.4f}', 'Train F1': f'{arr(name, "train_f1").mean():.4f}',
                 'Gap': f'{arr(name, "gap").mean():.4f} +/- {arr(name, "gap").std(ddof=1):.4f}',
                 'Benign FPR': f'{arr(name, "benign_fpr").mean():.4f}', 'Attack miss': f'{arr(name, "attack_miss_rate").mean():.4f}'})
summary = pd.DataFrame(rows); print(summary.to_string(index=False))

d = arr('tuned', 'test_f1') - arr('baseline_same_backend', 'test_f1')
tt = stats.ttest_rel(arr('tuned', 'test_f1'), arr('baseline_same_backend', 'test_f1'))
print(f'\nPaired difference (tuned - baseline), same backend & seeds: {d.mean():+.4f} +/- {d.std(ddof=1):.4f}  '
      f'(per seed: {np.round(d, 4).tolist()})')
print(f'Paired t-test: t={tt.statistic:.3f}, p={tt.pvalue:.4f}  (n={len(SEEDS)}; Wilcoxon not used -- its minimum two-sided p at n=5 is 0.0625)')
pooled = np.sqrt((arr('tuned', 'test_f1').var(ddof=1) + arr('baseline_same_backend', 'test_f1').var(ddof=1)) / 2)
print(f'Cohen d (pooled seed std): {d.mean() / pooled:.2f}')
verdict = ('TUNED BETTER' if (tt.pvalue < 0.05 and d.mean() > 0) else ('TUNED WORSE' if (tt.pvalue < 0.05 and d.mean() < 0) else 'NO SIGNIFICANT DIFFERENCE'))
print(f'\nVerdict: {verdict}')
if verdict == 'NO SIGNIFICANT DIFFERENCE':
    print('=> Report the tuned and untuned models as tied on clean accuracy; justify the choice of the tuned configuration on other grounds '
          '(smaller train-test gap, lower seed variance) only if the numbers above actually show them.')

# NB1 reference (CPU baseline)
try:
    nb1 = joblib.load(f'baseline_results_{PIPELINE_VERSION}.pkl')['multiseed']['XGBoost']['test_f1']
    print(f'\nReference -- NB1 CPU baseline (5 seeds): {nb1["mean"]:.4f} +/- {nb1["std"]:.4f}   '
          f'vs same-backend baseline here: {arr("baseline_same_backend", "test_f1").mean():.4f}  (difference = backend effect: '
          f'{arr("baseline_same_backend", "test_f1").mean() - nb1["mean"]:+.4f})')
except Exception as e:
    nb1 = None; print('NB1 reference not available:', e)

print('\nPer-class F1 (mean over 5 seeds):')
pc = pd.DataFrame({n: np.stack([RESULTS[n][s]['per_class_f1'] for s in SEEDS]).mean(0) for n in RESULTS}, index=class_names)
print(pc.round(4).T.to_string())

## 8. Detailed evaluation of the tuned model (seed 42)

In [ ]:
m42 = MODELS[('tuned', 42)]; pred = m42.predict(X_test)
print(classification_report(y_test, pred, target_names=class_names, digits=4))
print(pd.DataFrame(confusion_matrix(y_test, pred), index=class_names, columns=class_names))

## 9. Save to `SmartGrid_Dissertation`

* `optuna_xgboost_results_v6.pkl` -- tuned params, CV numbers, 5-seed tuned/baseline results, paired statistics.
* `xgb_tuned_model_v6.json` -- tuned model, seed 42 (the one downstream notebooks attack).
* `xgb_tuned_best_params_v6.json`, `optuna_xgb_trials_v6.csv`, `optuna_xgb_diagnostics_v6.png`.

In [ ]:
clean = {n: {s: {k: (v.tolist() if hasattr(v, 'tolist') else v) for k, v in r.items()} for s, r in d_.items()} for n, d_ in RESULTS.items()}
joblib.dump({
    'PIPELINE_VERSION': PIPELINE_VERSION, 'BACKEND': BACKEND_NAME, 'SEEDS': SEEDS, 'N_TRIALS': N_TRIALS, 'N_CV_FOLDS': N_CV_FOLDS,
    'best_params': study.best_params, 'best_cv_f1': study.best_value, 'baseline_cv_f1': base_cv_mean,
    'best_trial_fold_scores': study.best_trial.user_attrs['fold_scores'], 'baseline_fold_scores': base_cv_folds,
    'results': clean, 'summary_table': summary,
    'paired': {'mean_diff': float(d.mean()), 'std_diff': float(d.std(ddof=1)), 'per_seed_diff': d.tolist(),
               'p_value': float(tt.pvalue), 'verdict': verdict},
    'nb1_cpu_baseline': nb1,
    # legacy keys read by earlier notebooks
    'best_val_f1': study.best_value, 'test_f1': float(arr('tuned', 'test_f1')[0]), 'train_f1': float(arr('tuned', 'train_f1')[0]),
    'gap': float(arr('tuned', 'gap')[0]),
}, f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
MODELS[('tuned', 42)].save_model(f'xgb_tuned_model_{PIPELINE_VERSION}.json')
json.dump(study.best_params, open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json', 'w'), indent=2)
json.dump({'notebook': 'NB2', 'pipeline_version': PIPELINE_VERSION, 'backend': BACKEND_NAME, 'xgboost': xgb.__version__, 'optuna': optuna.__version__,
           'python': platform.python_version(), 'n_trials': N_TRIALS, 'seeds': SEEDS, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())},
          open(f'nb2_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl', f'xgb_tuned_model_{PIPELINE_VERSION}.json', f'xgb_tuned_best_params_{PIPELINE_VERSION}.json',
          f'optuna_xgb_trials_{PIPELINE_VERSION}.csv', f'optuna_xgb_diagnostics_{PIPELINE_VERSION}.png', f'nb2_run_manifest_{PIPELINE_VERSION}.json']:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)